In [ ]:
# ============================================================
# FedProx on Tomato Leaf Disease Dataset — Kaggle Notebook
# Conditions: ResNet18, SGD(lr=0.001, momentum=0.9),
#             batch=32, local_epochs=5, rounds=50,
#             5 clients, Dirichlet α=0.5, μ=0.01
# Dataset: https://www.kaggle.com/datasets/kaustubhb999/tomatoleaf
# ============================================================

# ── Cell 0: Verify dataset path ──────────────────────────────
import os

base = "/kaggle/input/tomatoleaf"
print("Scanning dataset structure...")
for root, dirs, files in os.walk(base):
    level = root.replace(base, '').count(os.sep)
    if level > 3:
        continue
    indent = '  ' * level
    n_files = len(files)
    print(f"{indent}{os.path.basename(root)}/"
          + (f"  [{n_files} files]" if n_files else ""))

# ── Cell 1: Imports ──────────────────────────────────────────
import copy
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import (precision_score, recall_score,
                             f1_score, accuracy_score)
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings("ignore")

# ── Cell 2: Reproducibility & Device ────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

try:
    DEVICE = torch.device("cuda")
    _t = torch.zeros(2, 2).to(DEVICE) + 1
    del _t
    print(f"CUDA OK — {torch.cuda.get_device_name(0)}")
except Exception as e:
    print(f"CUDA unavailable ({e}), using CPU")
    DEVICE = torch.device("cpu")

# ── Cell 3: Hyperparameters ──────────────────────────────────
NUM_CLIENTS     = 5
DIRICHLET_ALPHA = 0.5
LOCAL_EPOCHS    = 5
GLOBAL_ROUNDS   = 10
BATCH_SIZE      = 32
LR              = 0.001
MOMENTUM        = 0.9
MU              = 0.01        # FedProx proximal coefficient

# ── Cell 4: Dataset path ─────────────────────────────────────
# This dataset has structure: tomatoleaf/tomato/train/<class>/
# We merge train + test into one pool, then do our own split
TRAIN_DIR = "/kaggle/input/datasets/kaustubhb999/tomatoleaf/tomato/train"
TEST_DIR  = "/kaggle/input/datasets/kaustubhb999/tomatoleaf/tomato/val"   # 'val' or 'test'

# Auto-detect test folder name
if not os.path.exists(TEST_DIR):
    TEST_DIR = "/kaggle/input/tomatoleaf/tomato/test"
if not os.path.exists(TEST_DIR):
    # Some versions only have train
    TEST_DIR = None

print(f"Train dir : {TRAIN_DIR}  exists={os.path.exists(TRAIN_DIR)}")
print(f"Test  dir : {TEST_DIR}   exists={os.path.exists(TEST_DIR) if TEST_DIR else False}")

# ── Cell 5: Transforms ───────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# ── Cell 6: Load dataset (merge train+test into one pool) ────
train_dataset = datasets.ImageFolder(root=TRAIN_DIR)
NUM_CLASSES   = len(train_dataset.classes)
print(f"\nClasses ({NUM_CLASSES}): {train_dataset.classes}")
print(f"Train images: {len(train_dataset)}")

all_samples = list(train_dataset.samples)   # list of (path, label)

if TEST_DIR and os.path.exists(TEST_DIR):
    test_dataset = datasets.ImageFolder(root=TEST_DIR)
    # Remap test labels using train's class_to_idx for consistency
    remap = {v: train_dataset.class_to_idx[k]
             for k, v in test_dataset.class_to_idx.items()
             if k in train_dataset.class_to_idx}
    for path, lbl in test_dataset.samples:
        if lbl in remap:
            all_samples.append((path, remap[lbl]))
    print(f"Test  images: {len(test_dataset)}")

print(f"Total pooled : {len(all_samples)}")

# ── Cell 7: Lightweight dataset from sample list ─────────────
from PIL import Image

class SampleDataset(Dataset):
    """Dataset built from a list of (path, label) tuples."""
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

# ── Cell 8: Non-IID Dirichlet split ──────────────────────────
def dirichlet_split(samples, num_clients, alpha, num_classes, seed=SEED):
    np.random.seed(seed)
    labels = np.array([s[1] for s in samples])
    client_indices = [[] for _ in range(num_clients)]

    for cls in range(num_classes):
        cls_idx = np.where(labels == cls)[0]
        np.random.shuffle(cls_idx)
        if len(cls_idx) == 0:
            continue
        props = np.random.dirichlet(alpha * np.ones(num_clients))
        props = (props * len(cls_idx)).astype(int)
        diff  = len(cls_idx) - props.sum()
        props[np.argmax(props)] += diff
        splits = np.split(cls_idx, np.cumsum(props)[:-1])
        for c, split in enumerate(splits):
            client_indices[c].extend(split.tolist())

    for c in range(num_clients):
        random.shuffle(client_indices[c])

    return client_indices

client_indices = dirichlet_split(all_samples, NUM_CLIENTS,
                                 DIRICHLET_ALPHA, NUM_CLASSES)

print("\nClient data distribution:")
for i, idx in enumerate(client_indices):
    labels = [all_samples[j][1] for j in idx]
    print(f"  Client {i+1}: {len(idx):>5} samples | "
          f"{len(set(labels))} classes")

# ── Cell 9: Train/val split per client ───────────────────────
def train_val_split(indices, val_ratio=0.2, seed=SEED):
    random.seed(seed)
    indices = list(indices)
    random.shuffle(indices)
    split = int(len(indices) * (1 - val_ratio))
    return indices[:split], indices[split:]

client_train_idx = []
client_val_idx   = []
for idx in client_indices:
    tr, va = train_val_split(idx)
    client_train_idx.append(tr)
    client_val_idx.append(va)

# ── Cell 10: DataLoaders ─────────────────────────────────────
def make_loaders(train_ids, val_ids, all_samples):
    tr_samples  = [all_samples[i] for i in train_ids]
    val_samples = [all_samples[i] for i in val_ids]
    tr_set  = SampleDataset(tr_samples,  transform=train_transform)
    val_set = SampleDataset(val_samples, transform=val_transform)
    tr_loader  = DataLoader(tr_set,  batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=2, pin_memory=True)
    return tr_loader, val_loader

client_loaders = []
for i in range(NUM_CLIENTS):
    tr_l, va_l = make_loaders(client_train_idx[i],
                               client_val_idx[i], all_samples)
    client_loaders.append((tr_l, va_l))
    print(f"Client {i+1}: train={len(client_train_idx[i])}, "
          f"val={len(client_val_idx[i])}")

# Global val loader
all_val_idx = []
for va in client_val_idx:
    all_val_idx.extend(va)
val_samples_global = [all_samples[i] for i in all_val_idx]
global_val_set     = SampleDataset(val_samples_global, transform=val_transform)
global_val_loader  = DataLoader(global_val_set, batch_size=BATCH_SIZE,
                                shuffle=False, num_workers=2, pin_memory=True)
print(f"\nGlobal val set size: {len(all_val_idx)}")

# ── Cell 11: Model (ResNet18) ────────────────────────────────
def build_model(num_classes):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(DEVICE)

global_model = build_model(NUM_CLASSES)
print(f"ResNet18 → {NUM_CLASSES} output classes")

# ── Cell 12: FedProx local training ──────────────────────────
def fedprox_local_train(local_model, global_params, train_loader,
                        local_epochs, lr, momentum, mu):
    local_model.train()
    optimizer = optim.SGD(local_model.parameters(),
                          lr=lr, momentum=momentum)
    criterion = nn.CrossEntropyLoss()

    for _ in range(local_epochs):
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs  = local_model(images)
            ce_loss  = criterion(outputs, labels)
            # Proximal term: (μ/2) * ||w - w_global||²
            prox = sum(((p - g.detach()) ** 2).sum()
                       for p, g in zip(local_model.parameters(),
                                       global_params))
            loss = ce_loss + (mu / 2.0) * prox
            loss.backward()
            optimizer.step()

    return local_model.state_dict()

# ── Cell 13: FedAvg aggregation ──────────────────────────────
def fedavg_aggregate(global_model, client_state_dicts, client_sizes):
    total     = sum(client_sizes)
    avg_state = copy.deepcopy(client_state_dicts[0])
    for key in avg_state:
        avg_state[key] = torch.zeros_like(avg_state[key],
                                          dtype=torch.float32)
    for state, size in zip(client_state_dicts, client_sizes):
        w = size / total
        for key in avg_state:
            avg_state[key] += state[key].float() * w
    global_model.load_state_dict(avg_state)
    return global_model

# ── Cell 14: Evaluation ───────────────────────────────────────
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images  = images.to(DEVICE)
            outputs = model(images)
            preds   = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    acc  = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds,
                           average='weighted', zero_division=0)
    rec  = recall_score(all_labels, all_preds,
                        average='weighted', zero_division=0)
    f1   = f1_score(all_labels, all_preds,
                    average='weighted', zero_division=0)
    return acc, prec, rec, f1

# ── Cell 15: FedProx Training Loop ───────────────────────────
history = {"round": [], "accuracy": [],
           "precision": [], "recall": [], "f1": []}
best_acc, best_state = 0.0, None

print("\n" + "="*65)
print(f"{'FedProx Training — Tomato Leaf Disease':^65}")
print(f"  Clients={NUM_CLIENTS} | α={DIRICHLET_ALPHA} | μ={MU}")
print(f"  Rounds={GLOBAL_ROUNDS} | LocalEpochs={LOCAL_EPOCHS} | LR={LR}")
print("="*65)

for rnd in range(1, GLOBAL_ROUNDS + 1):

    global_params = [p.clone().detach()
                     for p in global_model.parameters()]
    client_states, client_sizes = [], []

    for c in range(NUM_CLIENTS):
        local_model = copy.deepcopy(global_model)
        tr_loader, _ = client_loaders[c]
        local_state  = fedprox_local_train(
            local_model, global_params, tr_loader,
            LOCAL_EPOCHS, LR, MOMENTUM, MU
        )
        client_states.append(local_state)
        client_sizes.append(len(client_train_idx[c]))

    global_model = fedavg_aggregate(global_model,
                                    client_states, client_sizes)

    acc, prec, rec, f1 = evaluate(global_model, global_val_loader)
    history["round"].append(rnd)
    history["accuracy"].append(acc)
    history["precision"].append(prec)
    history["recall"].append(rec)
    history["f1"].append(f1)

    if acc > best_acc:
        best_acc   = acc
        best_state = copy.deepcopy(global_model.state_dict())

    if rnd % 5 == 0 or rnd == 1:
        print(f"Round {rnd:>3}/{GLOBAL_ROUNDS} | "
              f"Acc={acc:.4f} | Prec={prec:.4f} | "
              f"Rec={rec:.4f} | F1={f1:.4f}")

print("\n" + "="*65)
print(f"Best Accuracy: {best_acc:.4f}  ({best_acc*100:.2f}%)")
print("="*65)

torch.save(best_state, "fedprox_tomatoleaf_best.pth")
print("Best model saved → fedprox_tomatoleaf_best.pth")

# ── Cell 16: Final evaluation ─────────────────────────────────
global_model.load_state_dict(best_state)
final_acc, final_prec, final_rec, final_f1 = evaluate(
    global_model, global_val_loader)

print("\n── Final Results (Best Model) ──────────────────────────")
print(f"  Accuracy  : {final_acc:.4f}  ({final_acc*100:.2f}%)")
print(f"  Precision : {final_prec:.4f}")
print(f"  Recall    : {final_rec:.4f}")
print(f"  F1-Score  : {final_f1:.4f}")
print("────────────────────────────────────────────────────────")

# ── Cell 17: Plots ────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    "FedProx — Tomato Leaf Disease (ResNet18)\n"
    f"Clients={NUM_CLIENTS}, α={DIRICHLET_ALPHA}, μ={MU}, "
    f"Rounds={GLOBAL_ROUNDS}, LocalEpochs={LOCAL_EPOCHS}",
    fontsize=13, fontweight='bold')

metrics = [
    ("accuracy",  "Accuracy",  "royalblue"),
    ("precision", "Precision", "darkorange"),
    ("recall",    "Recall",    "green"),
    ("f1",        "F1-Score",  "red"),
]
rounds = history["round"]
for ax, (key, label, color) in zip(axes.flatten(), metrics):
    ax.plot(rounds, history[key], color=color, linewidth=1.8)
    ax.set_title(label, fontsize=11)
    ax.set_xlabel("Communication Round")
    ax.set_ylabel(label)
    ax.set_xlim(1, GLOBAL_ROUNDS)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))
    ax.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig("fedprox_tomatoleaf_metrics.png", dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved → fedprox_tomatoleaf_metrics.png")

# ── Cell 18: Per-client performance ──────────────────────────
print("\n── Per-Client Validation Performance (Best Model) ──────")
for c in range(NUM_CLIENTS):
    _, val_loader = client_loaders[c]
    acc, prec, rec, f1 = evaluate(global_model, val_loader)
    print(f"  Client {c+1}: Acc={acc:.4f} | Prec={prec:.4f} | "
          f"Rec={rec:.4f} | F1={f1:.4f}")

# ── Cell 19: Config summary ───────────────────────────────────
print("\n── Experiment Configuration ─────────────────────────────")
for k, v in {
    "Dataset"            : "Tomato Leaf Disease (kaustubhb999)",
    "Model"              : "ResNet18",
    "FL Algorithm"       : "FedProx",
    "Num Clients"        : NUM_CLIENTS,
    "Dirichlet Alpha"    : DIRICHLET_ALPHA,
    "Proximal Coeff (μ)" : MU,
    "Local Epochs"       : LOCAL_EPOCHS,
    "Global Rounds"      : GLOBAL_ROUNDS,
    "Batch Size"         : BATCH_SIZE,
    "Learning Rate"      : LR,
    "Momentum"           : MOMENTUM,
    "Num Classes"        : NUM_CLASSES,
    "Total Samples"      : len(all_samples),
    "Best Accuracy"      : f"{best_acc*100:.2f}%",
    "Final F1-Score"     : f"{final_f1:.4f}",
}.items():
    print(f"  {k:<22}: {v}")